# Download 2024 ACS 5-year census tract data from the API

  - Available data: https://api.census.gov/data.html
  - 2024 ACS 5-year data profile documentation: https://www.census.gov/data/developers/data-sets/acs-5year/2024.html
  - Variables (fields) you can query: https://api.census.gov/data/2024/acs/acs5/profile/variables.html
  - Browse data profiles for a place: https://www.census.gov/acs/www/data/data-tables-and-tools/data-profiles/2024/
  - Sample query: https://api.census.gov/data/2024/acs/acs5/profile?get=NAME%2CDP05_0001E&for=tract%3A400100&in=state%3A06%20county%3A001&key=YOUR_KEY_GOES_HERE
  - Request an API key: https://api.census.gov/data/key_signup.html
  - County FIPS codes: https://www.census.gov/library/reference/code-lists/ansi.html
  - 2024 ACS geography files: https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.2024.html

#### Variables name format

More information:

  - Variable formats: https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-api-variable-formats.html
  - Variable types: https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-api-variable-types.html

variable name format: [TableID]_[RowNumber][VariableType]

Example: Variable DP02_0002PE, "Family households (families)", represents the percent estimate for table DP02 row number 2.

DP (Data Profile): Table type containing broad social, economic, housing, and demographic information in a total of four profiles.

  - DP02: Social Characteristics - includes Education, Marital Status, Relationships, Fertility, Grandparents...
  - DP03: Economic Characteristics - includes Income, Employment, Occupation, Commuting to Work...
  - DP04: Housing Characteristics - includes Occupancy and Structure, Housing Value and Costs, Utilities...
  - DP05: Demographic Characteristics - includes Sex and Age, Race, Hispanic Origin, Housing Units...

Variable suffixes:

  - E = estimate
  - M = margin of error
  - PE = percent estimate (of total)
  - PM = margin of error for corresponding PE
  - EA = estimate annotation
  - MA = margin-of-error annotation

Variables can change between ACS releases! Validate them again before you use a different release year.

In [ ]:
!curl -LsSf https://raw.githubusercontent.com/gboeing/ppd534/refs/heads/main/software/colab-installer.sh | bash

In [ ]:
import cenpy

API_KEY = "YOUR_API_KEY_HERE"

In [ ]:
# get 2024 ACS median household income data for all tracts in LA county
variables = ["DP03_0062E"]
acs = cenpy.remote.APIConnection("ACSDP5Y2024", apikey=API_KEY)
df = acs.query(cols=variables, geo_unit="tract:*", geo_filter={"state": "06", "county": "037"})
df["GEOID"] = df["state"] + df["county"] + df["tract"]
df.head()

In [ ]:
# or, get lots of variables at the same time
variables = [
    "DP05_0001E",  # total pop
    "DP05_0018E",  # median age
    "DP05_0009PE",  # pct pop aged 20-24
    "DP05_0010PE",  # pct pop aged 25-34
    "DP05_0024PE",  # pct pop age 65+
    "DP05_0090PE",  # pct pop hispanic or latino
    "DP05_0096PE",  # pct pop non-hispanic white alone
    "DP05_0097PE",  # pct pop non-hispanic black alone
    "DP05_0061PE",  # pct pop asian alone
    "DP05_0002PE",  # pct pop male
    "DP04_0007PE",  # pct single family detached homes
    "DP04_0089E",  # median value of owner occupied units (dollars)
    "DP04_0037E",  # median number of rooms in house
    "DP04_0026PE",  # pct structure built 1939 or earlier
    "DP04_0047PE",  # pct renter-occupied housing units
    "DP04_0047E",  # count renter-occupied housing units
    "DP04_0005E",  # rental vacancy rate
    "DP04_0049E",  # average household size of renter-occupied housing units
    "DP04_0134E",  # median gross rent (dollars)
    "DP04_0141PE",  # pct with gross rent 30-34.9% of household income
    "DP04_0142PE",  # pct with gross rent 35% or more of household income
    "DP03_0062E",  # median household income
    "DP03_0025E",  # mean travel time to work
    "DP03_0019PE",  # pct commute drove alone
    "DP03_0128PE",  # pct people with income below poverty level
    "DP02_0058PE",  # pct of enrolled students in college or grad school
    "DP02_0021PE",  # pct population in households: unmarried partner
    "DP02_0024PE",  # pct population in households: other nonrelatives
    "DP02_0080PE",  # pct residence 1 year ago was same house
    "DP02_0068PE",  # pct bachelor's degree or higher
    "DP02_0113PE",  # pct with english only language spoken at home
    "DP02_0094PE",
]  # pct of population foreign born

In [ ]:
acs = cenpy.remote.APIConnection("ACSDP5Y2024", apikey=API_KEY)
df = acs.query(cols=variables, geo_unit="tract:*", geo_filter={"state": "06", "county": "037"})
df["GEOID"] = df["state"] + df["county"] + df["tract"]
df = df.set_index("GEOID")
df.head()

In [ ]:
# or, you can even add tract geometry polygons as well
tiger = cenpy.tiger.TigerConnection("tigerWMS_ACS2024")
tracts = tiger.query(
    layer=8,  # this is for census tracts
    where="STATE='06' AND COUNTY='037'",
    outFields="GEOID",
    outSR=4326,
)
gdf = tracts.set_index("GEOID").join(df, how="inner")
gdf.shape

In [ ]:
gdf.head()

In [ ]:
# or you can map the tracts by median household income
gdf["DP03_0062E"] = gdf["DP03_0062E"].astype(float)
gdf = gdf[gdf["DP03_0062E"] > 0]
gdf.plot(column="DP03_0062E")